In [ ]:
import os

import open3d as o3d


def is_stl_watertight_open3d(filepath):
    """
    Checks if an STL file is watertight using Open3D.

    Args:
        filepath (str): The path to the STL file.

    Returns:
        bool: True if the mesh is watertight, False otherwise.
        str: A message indicating the result or any errors.
    """
    if not os.path.exists(filepath):
        return False, f"Error: File not found at {filepath}"

    try:
        mesh = o3d.io.read_triangle_mesh(filepath)

        if not mesh.has_triangle_normals():
            mesh.compute_triangle_normals()
        if not mesh.has_vertex_normals():
            mesh.compute_vertex_normals()

        # Open3D's is_watertight checks for edge_manifold, vertex_manifold, and self_intersection
        is_watertight = mesh.is_watertight()

        if is_watertight:
            return True, f"The mesh in {filepath} is watertight."
        else:
            # You can further investigate why it's not watertight
            is_edge_manifold = mesh.is_edge_manifold(allow_boundary_edges=True)
            is_vertex_manifold = mesh.is_vertex_manifold()
            is_self_intersecting = mesh.is_self_intersecting()

            issues = []
            if not is_edge_manifold:
                issues.append(
                    "not edge manifold (has edges connected to more than two triangles)"
                )
            # Check with allow_boundary_edges=False to see if there are boundary edges (holes)
            if mesh.is_edge_manifold(allow_boundary_edges=False) and is_edge_manifold:
                pass  # It's edge manifold, and no boundary edges mean no holes
            elif not mesh.is_edge_manifold(allow_boundary_edges=False):
                issues.append("has boundary edges (holes)")

            if not is_vertex_manifold:
                issues.append(
                    "not vertex manifold (has vertices where multiple surfaces meet incorrectly)"
                )
            if is_self_intersecting:
                issues.append("has self-intersections")

            if issues:
                return (
                    False,
                    f"The mesh in {filepath} is NOT watertight. Issues found: {', '.join(issues)}",
                )
            else:
                return (
                    False,
                    f"The mesh in {filepath} is NOT watertight, but no specific common issues were detected by basic checks. It might have complex non-manifold geometry or other less common problems.",
                )

    except Exception as e:
        return False, f"Error processing file {filepath}: {e}"


is_watertight, message = is_stl_watertight_open3d(
    "/Users/nils/ace/workspace/aisco-ET/assembly/fused_wo_seat.stl"
)
print(message)

# Example with a (theoretically) watertight mesh (like a simple cube)
# You would replace 'path/to/your/watertight.stl' with your actual file
# is_watertight, message = is_stl_watertight_open3d("path/to/your/watertight.stl")
# print(message)

In [ ]:
def visualize_stl_open3d(filepath):
    """
    Loads and visualizes an STL file using Open3D.
    """
    try:
        mesh = o3d.io.read_triangle_mesh(filepath)
        if not mesh.has_triangle_normals():
            mesh.compute_triangle_normals()
        if not mesh.has_vertex_normals():
            mesh.compute_vertex_normals()
        o3d.visualization.draw_geometries([mesh])
    except Exception as e:
        print(f"Error loading or visualizing {filepath}: {e}")


visualize_stl_open3d("/Users/nils/ace/workspace/aisco-ET/assembly/fused_wo_seat.stl")

In [ ]:
def visualize_stl_wireframe(filepath):
    """
    Loads an STL file and visualizes its wireframe (triangles) using Open3D.

    Args:
        filepath (str): The path to the STL file.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return

    try:
        # Load the triangle mesh from the STL file
        mesh = o3d.io.read_triangle_mesh(filepath)

        # Ensure normals are computed (often needed for proper visualization)
        if not mesh.has_triangle_normals():
            mesh.compute_triangle_normals()
        if not mesh.has_vertex_normals():
            mesh.compute_vertex_normals()

        # Create a visualization object and set rendering options
        vis = o3d.visualization.Visualizer()
        vis.create_window()

        # Add the mesh to the visualizer
        vis.add_geometry(mesh)

        # Set rendering options to show wireframe
        # Get the rendering options object
        render_option = vis.get_render_option()
        # Set mesh_show_wireframe to True
        render_option.mesh_show_wireframe = True
        # Optionally, set mesh_show_back_face to True to see back-facing triangles
        # render_option.mesh_show_back_face = True # Can help spot flipped normals

        # Run the visualizer
        vis.run()
        vis.destroy_window()

    except Exception as e:
        print(f"Error loading or visualizing {filepath}: {e}")


visualize_stl_wireframe("/Users/nils/ace/workspace/aisco-ET/assembly/fused.stl")